# Feedforward Pointcloud Reconstruction

This tutorial runs **VGGT-X**, **MapAnything**, and **VGGT-Omega** on keyframes extracted from a real video, then compares the resulting pointclouds side by side. All three models run feedforward inference (no optimisation loop) and write results to zarr caches that downstream notebooks — `semantic_lifting`, `localization`, and `bundle_adjustment` — depend on. **Run this notebook before any of those.**

**Prerequisite:** [Keyframe Extraction](../01_preprocessing/keyframe_extraction.ipynb) must have been executed first to populate the `images/` directory used here.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import open3d as o3d
import pyvista as pv
import torch
%matplotlib inline

# pv.set_jupyter_backend("static" if os.environ.get("PYVISTA_OFF_SCREEN") else "trame")
pv.set_jupyter_backend("trame")

from collab_splats.pointcloud.feedforward import VGGTXCreator, MapAnythingCreator, VGGTOmegaCreator
from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.pointcloud.utils import clean_pointcloud
from collab_splats.utils.visualization import pointcloud_to_polydata, visualize_splat, create_camera_frustum_pyvista

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────

assert IMAGES.exists() and any(IMAGES.glob("*.jpg")), (
    f"No images found in {IMAGES}. Run 01_preprocessing/keyframe_extraction first."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}  |  images: {len(list(IMAGES.glob('*.jpg')))}")

## §1 — VGGT-X Reconstruction

Loads a cached VGGT-X reconstruction from zarr if available, skipping GPU inference (~3–5 min on GPU). If no cache exists, runs VGGT-X on the keyframes and saves the result. The zarr file is chunked by frame for efficient per-frame random access by downstream notebooks (semantic_lifting, localization).

In [ ]:
_vggtx_cache = CACHE_DIR / "vggtx" / "reconstruction.zarr"
_vggtx_cache.parent.mkdir(parents=True, exist_ok=True)

if _vggtx_cache.exists():
    result_vggt = FeedforwardResult.load_zarr(_vggtx_cache)
    print(f"Loaded VGGT-X result from cache  ({result_vggt.points.shape[0]:,} pts)")
else:
    result_vggt = VGGTXCreator().run(IMAGES, device)
    result_vggt.save_zarr(_vggtx_cache)
    print(f"VGGT-X done  →  saved to {_vggtx_cache}")

## §2 — VGGT-X Post-processing and Visualisation

Removes statistical outliers and downsamples via voxel grid to produce a clean pointcloud. Confidence statistics are printed to help diagnose prediction quality.

In [ ]:
# Clean via Open3D statistical outlier removal + voxel downsampling
_pcd = o3d.geometry.PointCloud()
_pcd.points = o3d.utility.Vector3dVector(result_vggt.points)
_pcd.colors = o3d.utility.Vector3dVector(result_vggt.colors.astype(np.float64) / 255.0)
_cleaned, _ = clean_pointcloud(_pcd)

pts3d_vggt = np.asarray(_cleaned.points, dtype=np.float32)
colors_vggt = (np.asarray(_cleaned.colors) * 255).astype(np.uint8)

conf_vggt_mean = result_vggt.confidence.cpu().float().mean().item() if result_vggt.confidence is not None else float("nan")
conf_vggt_std  = result_vggt.confidence.cpu().float().std().item()  if result_vggt.confidence is not None else float("nan")

print(f"Points: {len(result_vggt.points):,} raw → {len(pts3d_vggt):,} filtered")
print(f"Conf:   mean={conf_vggt_mean:.3f}  std={conf_vggt_std:.3f}")

## §3 — VGGT-X Pointcloud Viewer

Renders the filtered VGGT-X pointcloud with camera frustums overlaid. Blue frustums show predicted camera poses.

In [ ]:
cloud_vggt = pointcloud_to_polydata(pts3d_vggt, RGB=colors_vggt)
c2w_vggt = [np.linalg.inv(ext) for ext in result_vggt.extrinsics]
pl = visualize_splat(cloud_vggt, aligned_cameras=c2w_vggt)
pl.show()

## §4 — MapAnything Reconstruction

MapAnything predicts dense depth and camera poses via cross-view consistency.
Uses `confidence_percentile` to mask unreliable pixels before unprojection.

In [ ]:
_ma_cache = CACHE_DIR / "mapanything" / "reconstruction.zarr"
_ma_cache.parent.mkdir(parents=True, exist_ok=True)

if _ma_cache.exists():
    result_ma = FeedforwardResult.load_zarr(_ma_cache)
    print(f"Loaded MapAnything result from cache  ({result_ma.points.shape[0]:,} pts)")
else:
    result_ma = MapAnythingCreator().run(IMAGES, device)
    result_ma.save_zarr(_ma_cache)
    print(f"MapAnything done  →  saved to {_ma_cache}")

## §5 — MapAnything Post-processing and Visualisation

Applies the same outlier removal and voxel downsampling pipeline as VGGT-X. Confidence metrics are collected for the side-by-side comparison table.

In [ ]:
# Clean via Open3D statistical outlier removal + voxel downsampling
_pcd = o3d.geometry.PointCloud()
_pcd.points = o3d.utility.Vector3dVector(result_ma.points)
_pcd.colors = o3d.utility.Vector3dVector(result_ma.colors.astype(np.float64) / 255.0)
_cleaned, _ = clean_pointcloud(_pcd)

pts3d_ma = np.asarray(_cleaned.points, dtype=np.float32)
colors_ma = (np.asarray(_cleaned.colors) * 255).astype(np.uint8)

conf_ma_mean = result_ma.confidence.cpu().float().mean().item() if result_ma.confidence is not None else float("nan")
conf_ma_std  = result_ma.confidence.cpu().float().std().item()  if result_ma.confidence is not None else float("nan")

print(f"Points: {len(result_ma.points):,} raw → {len(pts3d_ma):,} filtered")
print(f"Conf:   mean={conf_ma_mean:.3f}  std={conf_ma_std:.3f}")

## §6 — MapAnything Pointcloud Viewer

Renders the filtered MapAnything pointcloud with camera frustums. Orange frustums show predicted camera poses; compare pose spread vs. VGGT-X above.

In [ ]:
cloud_ma = pointcloud_to_polydata(pts3d_ma, RGB=colors_ma)
c2w_ma = [np.linalg.inv(ext) for ext in result_ma.extrinsics]
pl = visualize_splat(cloud_ma, aligned_cameras=c2w_ma)
pl.show()

## §9 — VGGT-Omega Reconstruction

Loads a cached VGGT-Omega reconstruction from zarr if available, skipping GPU inference (~3–5 min on GPU). VGGT-Omega is a third feedforward method that uses a unified vision transformer for depth and pose estimation. Runs with default settings (`VGGTOmegaCreator()`).

In [ ]:
_omega_cache = CACHE_DIR / "omega" / "reconstruction.zarr"
_omega_cache.parent.mkdir(parents=True, exist_ok=True)

if _omega_cache.exists():
    result_omega = FeedforwardResult.load_zarr(_omega_cache)
    print(f"Loaded VGGT-Omega result from cache  ({result_omega.points.shape[0]:,} pts)")
else:
    result_omega = VGGTOmegaCreator().run(IMAGES, device)
    result_omega.save_zarr(_omega_cache)
    print(f"VGGT-Omega done  →  saved to {_omega_cache}")

## §10 — VGGT-Omega Post-processing and Visualisation

Applies the same outlier removal and voxel downsampling pipeline as §2/§5. Confidence metrics are collected for the three-way comparison table.

In [ ]:
# Clean via Open3D statistical outlier removal + voxel downsampling
_pcd = o3d.geometry.PointCloud()
_pcd.points = o3d.utility.Vector3dVector(result_omega.points)
_pcd.colors = o3d.utility.Vector3dVector(result_omega.colors.astype(np.float64) / 255.0)
_cleaned, _ = clean_pointcloud(_pcd)

pts3d_omega = np.asarray(_cleaned.points, dtype=np.float32)
colors_omega = (np.asarray(_cleaned.colors) * 255).astype(np.uint8)

conf_omega_mean = result_omega.confidence.cpu().float().mean().item() if result_omega.confidence is not None else float("nan")
conf_omega_std  = result_omega.confidence.cpu().float().std().item()  if result_omega.confidence is not None else float("nan")

print(f"Points: {len(result_omega.points):,} raw → {len(pts3d_omega):,} filtered")
print(f"Conf:   mean={conf_omega_mean:.3f}  std={conf_omega_std:.3f}")

## §11 — VGGT-Omega Pointcloud Viewer

Renders the filtered VGGT-Omega pointcloud with camera frustums. Camera frustums show predicted camera poses; compare pose spread vs. VGGT-X (§3) and MapAnything (§6) above.

In [ ]:
cloud_omega = pointcloud_to_polydata(pts3d_omega, RGB=colors_omega)
c2w_omega = [np.linalg.inv(ext) for ext in result_omega.extrinsics]
pl = visualize_splat(cloud_omega, aligned_cameras=c2w_omega)
pl.show()

## §12 — Side-by-side Comparison

Tabulates raw vs. filtered point counts and confidence statistics for all three models. Higher confidence mean with lower std indicates more reliable depth predictions.

In [ ]:
col = 14
print(f"{'Model':<{col}} {'Pts raw':>10} {'Pts filt':>10} {'Conf mean':>10} {'Conf std':>10}")
print("-" * (col + 42))
print(f"{'VGGT-X':<{col}} {len(result_vggt.points):>10,} {len(pts3d_vggt):>10,} {conf_vggt_mean:>10.3f} {conf_vggt_std:>10.3f}")
print(f"{'MapAnything':<{col}} {len(result_ma.points):>10,} {len(pts3d_ma):>10,} {conf_ma_mean:>10.3f} {conf_ma_std:>10.3f}")
print(f"{'VGGT-Omega':<{col}} {len(result_omega.points):>10,} {len(pts3d_omega):>10,} {conf_omega_mean:>10.3f} {conf_omega_std:>10.3f}")

## §13 — Camera Pose Overlay

Overlays camera frustums from all three models in a single scene — blue for VGGT-X, orange for MapAnything, green for VGGT-Omega. Alignment across the three sets indicates consistent global pose estimation.

In [ ]:
pl = pv.Plotter()
for ext in result_vggt.extrinsics:
    frustum = create_camera_frustum_pyvista(np.linalg.inv(ext), scale=0.05)
    pl.add_mesh(frustum, color="cornflowerblue", line_width=2)
for ext in result_ma.extrinsics:
    frustum = create_camera_frustum_pyvista(np.linalg.inv(ext), scale=0.05)
    pl.add_mesh(frustum, color="darkorange", line_width=2)
for ext in result_omega.extrinsics:
    frustum = create_camera_frustum_pyvista(np.linalg.inv(ext), scale=0.05)
    pl.add_mesh(frustum, color="mediumseagreen", line_width=2)
pl.add_axes()
pl.show()